In [10]:
import pandas as pd
df = pd.read_csv("malnutrition_children_ethiopia_balanced.csv")
print("Shape:", df.shape)
print("Columns:")
print(df.columns)
print("First 5 rows:")
print(df.head())
print("Data types:")
print(df.dtypes)

Shape: (2490, 16)
Columns:
Index(['ID', 'Age (months)', 'Gender', 'Region', 'Mother_Education',
       'Household_Wealth_Index', 'Height_cm', 'Weight_kg', 'Stunting',
       'Underweight', 'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB',
       'Nutrition_Status'],
      dtype='object')
First 5 rows:
     ID  Age (months)  Gender       Region Mother_Education  \
0  2064            54  Female       Tigray          Primary   
1   612            43    Male  Addis Ababa           Higher   
2  2972            56  Female       Amhara        Secondary   
3  3205            18  Female       Amhara        Secondary   
4  3327             8  Female       Oromia     No education   

  Household_Wealth_Index  Height_cm  Weight_kg  Stunting  Underweight  \
0                    Low       63.7       14.3         1            0   
1                 Middle       89.1       13.9         1            0   
2                   High       90.1       17.4         1            1   
3                    Lo

In [11]:
y = df["Nutrition_Status"]
X = df.drop(columns=["ID", "Nutrition_Status"])

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget class distribution:")
print(y.value_counts())

Features shape: (2490, 14)
Target shape: (2490,)

Target class distribution:
Nutrition_Status
At_Risk         830
Malnourished    830
Normal          830
Name: count, dtype: int64


In [48]:
X_encoded = pd.get_dummies(
    X,
    columns=[
        "Gender",
        "Region",
        "Mother_Education",
        "Household_Wealth_Index"
    ],
    drop_first=True
)

print("Encoded feature shape:", X_encoded.shape)
print("\nEncoded feature columns:")
print(X_encoded.columns)

Encoded feature shape: (2490, 20)

Encoded feature columns:
Index(['Age (months)', 'Height_cm', 'Weight_kg', 'Stunting', 'Underweight',
       'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB', 'Gender_Male',
       'Region_Amhara', 'Region_Oromia', 'Region_SNNPR', 'Region_Tigray',
       'Mother_Education_No education', 'Mother_Education_Primary',
       'Mother_Education_Secondary', 'Household_Wealth_Index_Low',
       'Household_Wealth_Index_Middle'],
      dtype='object')


In [13]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Encoded target classes mapping:")
for cls, enc in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"{cls} -> {enc}")

print("\nFirst 10 encoded labels:")
print(y_encoded[:10])

Encoded target classes mapping:
At_Risk -> 0
Malnourished -> 1
Normal -> 2

First 10 encoded labels:
[0 0 0 0 0 0 0 0 0 0]


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining target distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest target distribution:")
print(pd.Series(y_test).value_counts())

Training set shape: (1992, 20)
Test set shape: (498, 20)

Training target distribution:
0    664
2    664
1    664
Name: count, dtype: int64

Test target distribution:
2    166
1    166
0    166
Name: count, dtype: int64


In [47]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Scaled training shape: (1992, 20)
Scaled test shape: (498, 20)


In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.3293172690763052

Classification Report:
              precision    recall  f1-score   support

           0       0.29      0.36      0.32       166
           1       0.33      0.28      0.30       166
           2       0.38      0.34      0.36       166

    accuracy                           0.33       498
   macro avg       0.33      0.33      0.33       498
weighted avg       0.33      0.33      0.33       498



In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.321285140562249

Classification Report:
              precision    recall  f1-score   support

           0       0.30      0.33      0.31       166
           1       0.30      0.28      0.29       166
           2       0.36      0.36      0.36       166

    accuracy                           0.32       498
   macro avg       0.32      0.32      0.32       498
weighted avg       0.32      0.32      0.32       498



In [18]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, y_pred_rf)

cm_df = pd.DataFrame(
    cm,
    index=["At_Risk (0)", "Malnourished (1)", "Normal (2)"],
    columns=["Pred At_Risk", "Pred Malnourished", "Pred Normal"]
)

print(cm_df)

                  Pred At_Risk  Pred Malnourished  Pred Normal
At_Risk (0)                 54                 58           54
Malnourished (1)            70                 47           49
Normal (2)                  55                 52           59


In [19]:

import numpy as np

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_encoded.columns
).sort_values(ascending=False)

print(feature_importance)

Height_cm                        0.173779
Weight_kg                        0.172015
Age (months)                     0.162065
Underweight                      0.035985
Diarrhea                         0.035887
Malaria                          0.035355
Overweight                       0.035268
Gender_Male                      0.035198
TB                               0.034769
Stunting                         0.031973
Household_Wealth_Index_Low       0.029551
Anemia                           0.029333
Mother_Education_Secondary       0.025391
Mother_Education_Primary         0.025356
Household_Wealth_Index_Middle    0.024746
Mother_Education_No education    0.024445
Region_SNNPR                     0.023267
Region_Tigray                    0.022642
Region_Amhara                    0.022454
Region_Oromia                    0.020522
dtype: float64


In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_tuned = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced"
)

rf_tuned.fit(X_train_scaled, y_train)

y_pred_rf_tuned = rf_tuned.predict(X_test_scaled)

print("Tuned Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_tuned))

Tuned Random Forest Accuracy: 0.3413654618473896

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.33      0.33       166
           1       0.34      0.36      0.35       166
           2       0.36      0.34      0.35       166

    accuracy                           0.34       498
   macro avg       0.34      0.34      0.34       498
weighted avg       0.34      0.34      0.34       498



In [21]:
y_binary = y_encoded.copy()
y_binary = (y_binary != 2).astype(int)

import pandas as pd
print(pd.Series(y_binary).value_counts())

1    1660
0     830
Name: count, dtype: int64


In [22]:
from sklearn.model_selection import train_test_split

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_encoded,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print("Training shape:", X_train_b.shape)
print("Test shape:", X_test_b.shape)

print("\nTraining target distribution:")
print(pd.Series(y_train_b).value_counts())

print("\nTest target distribution:")
print(pd.Series(y_test_b).value_counts())

Training shape: (1992, 20)
Test shape: (498, 20)

Training target distribution:
1    1328
0     664
Name: count, dtype: int64

Test target distribution:
1    332
0    166
Name: count, dtype: int64


In [23]:
from sklearn.preprocessing import StandardScaler

scaler_b = StandardScaler()

X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

print("Scaled training shape:", X_train_b_scaled.shape)
print("Scaled test shape:", X_test_b_scaled.shape)

Scaled training shape: (1992, 20)
Scaled test shape: (498, 20)


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

lr_b = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_b.fit(X_train_b_scaled, y_train_b)

y_pred_lr_b = lr_b.predict(X_test_b_scaled)


print("Binary Logistic Regression Accuracy:", accuracy_score(y_test_b, y_pred_lr_b))
print("\nClassification Report:")
print(classification_report(y_test_b, y_pred_lr_b))

Binary Logistic Regression Accuracy: 0.4819277108433735

Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.46      0.37       166
           1       0.65      0.49      0.56       332

    accuracy                           0.48       498
   macro avg       0.48      0.48      0.47       498
weighted avg       0.53      0.48      0.50       498



In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_b = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced"
)

rf_b.fit(X_train_b_scaled, y_train_b)


y_pred_rf_b = rf_b.predict(X_test_b_scaled)

print("Binary Random Forest Accuracy:", accuracy_score(y_test_b, y_pred_rf_b))
print("\nClassification Report:")
print(classification_report(y_test_b, y_pred_rf_b))

Binary Random Forest Accuracy: 0.6566265060240963

Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.02      0.04       166
           1       0.67      0.97      0.79       332

    accuracy                           0.66       498
   macro avg       0.49      0.50      0.42       498
weighted avg       0.55      0.66      0.54       498



In [26]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [27]:
neg = (y_train_b == 0).sum()
pos = (y_train_b == 1).sum()
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 0.5


In [28]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

xgb.fit(X_train_b_scaled, y_train_b)

y_pred_xgb = xgb.predict(X_test_b_scaled)

print("XGBoost Accuracy:", accuracy_score(y_test_b, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test_b, y_pred_xgb))

XGBoost Accuracy: 0.5261044176706827

Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.34      0.33       166
           1       0.65      0.62      0.63       332

    accuracy                           0.53       498
   macro avg       0.48      0.48      0.48       498
weighted avg       0.54      0.53      0.53       498



In [29]:
import numpy as np
from sklearn.metrics import accuracy_score

y_probs = xgb.predict_proba(X_test_b_scaled)[:, 1]

for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    y_pred_t = (y_probs >= t).astype(int)
    acc = accuracy_score(y_test_b, y_pred_t)
    print(f"Threshold {t}: Accuracy = {acc:.3f}")

Threshold 0.4: Accuracy = 0.610
Threshold 0.45: Accuracy = 0.562
Threshold 0.5: Accuracy = 0.526
Threshold 0.55: Accuracy = 0.498
Threshold 0.6: Accuracy = 0.476


In [30]:
import numpy as np

X_fe = X.copy()

X_fe["Height_m"] = X_fe["Height_cm"] / 100
X_fe["BMI"] = X_fe["Weight_kg"] / (X_fe["Height_m"] ** 2)

X_fe["Weight_per_month"] = X_fe["Weight_kg"] / (X_fe["Age (months)"] + 1)

X_fe["Height_per_month"] = X_fe["Height_cm"] / (X_fe["Age (months)"] + 1)

print(X_fe[["BMI", "Weight_per_month", "Height_per_month"]].describe())

               BMI  Weight_per_month  Height_per_month
count  2490.000000       2490.000000       2490.000000
mean     19.174937          0.957168          6.536187
std      10.096294          1.965871         12.863044
min       4.237961          0.083333          1.003333
25%      11.701147          0.251026          1.828220
50%      16.708704          0.402326          2.716954
75%      24.885040          0.756000          5.052801
max      54.444444         19.600000        108.600000


In [31]:
X_fe["Age_Group"] = pd.cut(
    X_fe["Age (months)"],
    bins=[-1, 6, 24, 60],
    labels=["0-6m", "6-24m", "24-60m"]
)

print(X_fe["Age_Group"].value_counts())

Age_Group
24-60m    1466
6-24m      758
0-6m       266
Name: count, dtype: int64


In [32]:
health_cols = ["Stunting", "Underweight", "Anemia", "Malaria", "Diarrhea", "TB"]

X_fe["Health_Burden"] = X_fe[health_cols].sum(axis=1)

print(X_fe["Health_Burden"].value_counts().sort_index())

Health_Burden
0     30
1    224
2    585
3    804
4    585
5    230
6     32
Name: count, dtype: int64


In [33]:
X_fe_encoded = pd.get_dummies(
    X_fe,
    columns=["Age_Group"],
    drop_first=True
)

print("New feature shape:", X_fe_encoded.shape)
print(X_fe_encoded.columns)


New feature shape: (2490, 21)
Index(['Age (months)', 'Gender', 'Region', 'Mother_Education',
       'Household_Wealth_Index', 'Height_cm', 'Weight_kg', 'Stunting',
       'Underweight', 'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB',
       'Height_m', 'BMI', 'Weight_per_month', 'Height_per_month',
       'Health_Burden', 'Age_Group_6-24m', 'Age_Group_24-60m'],
      dtype='object')


In [34]:
X_final = pd.get_dummies(
    X_fe_encoded,
    columns=[
        "Gender",
        "Region",
        "Mother_Education",
        "Household_Wealth_Index"
    ],
    drop_first=True
)

print("Final feature shape:", X_final.shape)

Final feature shape: (2490, 27)


In [35]:
from sklearn.model_selection import train_test_split

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_final,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print(X_train_f.shape, X_test_f.shape)

(1992, 27) (498, 27)


In [36]:
from sklearn.preprocessing import StandardScaler

scaler_f = StandardScaler()
X_train_f_scaled = scaler_f.fit_transform(X_train_f)
X_test_f_scaled = scaler_f.transform(X_test_f)

print(X_train_f_scaled.shape, X_test_f_scaled.shape)

(1992, 27) (498, 27)


In [37]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
rf_fe = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced"
)

rf_fe.fit(X_train_f_scaled, y_train_f)

y_pred_rf_fe = rf_fe.predict(X_test_f_scaled)

print("RF (engineered) Accuracy:", accuracy_score(y_test_f, y_pred_rf_fe))
print("\nClassification Report:")
print(classification_report(y_test_f, y_pred_rf_fe))

RF (engineered) Accuracy: 0.6646586345381527

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.06      0.11       166
           1       0.67      0.97      0.79       332

    accuracy                           0.66       498
   macro avg       0.57      0.51      0.45       498
weighted avg       0.61      0.66      0.56       498



In [38]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

neg = (y_train_f == 0).sum()
pos = (y_train_f == 1).sum()
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

xgb_fe = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

xgb_fe.fit(X_train_f_scaled, y_train_f)

y_pred_xgb_fe = xgb_fe.predict(X_test_f_scaled)

print("XGBoost (engineered) Accuracy:", accuracy_score(y_test_f, y_pred_xgb_fe))
print("\nClassification Report:")
print(classification_report(y_test_f, y_pred_xgb_fe))

scale_pos_weight: 0.5
XGBoost (engineered) Accuracy: 0.5200803212851406

Classification Report:
              precision    recall  f1-score   support

           0       0.30      0.33      0.32       166
           1       0.65      0.61      0.63       332

    accuracy                           0.52       498
   macro avg       0.47      0.47      0.47       498
weighted avg       0.53      0.52      0.53       498



In [39]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

In [40]:
from sklearn.model_selection import train_test_split

X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X_final,         
    y_encoded,         
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_train_cb.shape, X_test_cb.shape)

(1992, 27) (498, 27)


In [41]:
cb = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=500,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=False
)

cb.fit(X_train_cb, y_train_cb)

y_pred_cb = cb.predict(X_test_cb).astype(int).flatten()

print("CatBoost (3-class) Accuracy:", accuracy_score(y_test_cb, y_pred_cb))
print("\nClassification Report:")
print(classification_report(y_test_cb, y_pred_cb))


CatBoost (3-class) Accuracy: 0.3433734939759036

Classification Report:
              precision    recall  f1-score   support

           0       0.32      0.33      0.33       166
           1       0.34      0.34      0.34       166
           2       0.37      0.36      0.36       166

    accuracy                           0.34       498
   macro avg       0.34      0.34      0.34       498
weighted avg       0.34      0.34      0.34       498



In [42]:
from sklearn.model_selection import train_test_split

X_train_rf3, X_test_rf3, y_train_rf3, y_test_rf3 = train_test_split(
    X_final,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_train_rf3.shape, X_test_rf3.shape)

(1992, 27) (498, 27)


In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_3c = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,          
    min_samples_leaf=3,     
    class_weight="balanced",
    random_state=42
)

rf_3c.fit(X_train_rf3, y_train_rf3)

y_pred_rf3 = rf_3c.predict(X_test_rf3)

print("Random Forest (3-class) Accuracy:", accuracy_score(y_test_rf3, y_pred_rf3))
print("\nClassification Report:")
print(classification_report(y_test_rf3, y_pred_rf3))

Random Forest (3-class) Accuracy: 0.3453815261044177

Classification Report:
              precision    recall  f1-score   support

           0       0.35      0.37      0.36       166
           1       0.33      0.34      0.34       166
           2       0.35      0.32      0.34       166

    accuracy                           0.35       498
   macro avg       0.35      0.35      0.35       498
weighted avg       0.35      0.35      0.35       498

